# MotionJSON Local UI setup

This notebook's normal path is only to install MotionJSON, launch the same main Local UI used outside Colab, and open it inside the notebook. Do model configuration in the UI: choose a goal, import video, open **Model setup**, then use the inline install, access check, diagnose, and smoke-test actions for SAM3 Scene Sweep, SAM2 fallback, hosted SAM3, hosted SAM2, or custom endpoints.

Run the first launch cells, then stay in the UI. You do not need to run the local SAM2/SAM3 package, checkpoint, or diagnostics cells unless the UI tells you to use the advanced fallback path.

No public tunnel is started. Hosted provider keys are read from Colab userdata when available, with interactive fallback. You can also leave them blank and paste temporary credentials into the UI Model setup cards. Do not save private videos, provider credentials, SAM checkpoints, hosted-service keys, or shared notebook outputs containing secrets.

Colab GPU availability, memory, runtime length, and VM lifetime are not guaranteed. Provider settings and environment variables only mean a provider is configured; MotionJSON diagnostics must still show local SAM2/SAM3 or hosted SAM2/SAM3 as runnable before a real run should start. Hosted providers can send frames, prompts, or derived image data to third-party services and require explicit cost/privacy acknowledgement.

If you prefer hosted SAM3, configure it from Model setup. Roboflow SAM3 and Fal SAM3 image do not require local SAM3 package or checkpoint cells.

In the UI, use **Start**, **Video**, **Model setup**, **Prepare**, **Run**, and **Review & export**. Failed setup or extraction runs stay in the flow with **Open logs**, **Change setup**, **Run again**, and **Choose different model**.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_url = "https://github.com/ptse8204/json-animated-video.git"
workdir = Path("/content/json-animated-video")

if not workdir.exists():
    subprocess.run(["git", "clone", repo_url, str(workdir)], check=True)
else:
    subprocess.run(["git", "-C", str(workdir), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(workdir), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(workdir), "pull", "--ff-only"], check=True)

os.chdir(workdir)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[ui,hosted-segmentation,hosted-sam3,hosted-sam-vendors]"],
    check=True,
)


In [ ]:
import time
from google.colab import output

port = 8766
if 'ui_proc' in globals() and ui_proc.poll() is None:
    print("MotionJSON UI is already running on port", port)
else:
    ui_proc = subprocess.Popen(
        ["motionjson", "ui", "--no-open", "--host", "127.0.0.1", "--port", str(port)],
        cwd=str(workdir),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    time.sleep(5)
    print("MotionJSON UI started with: motionjson ui --no-open --host 127.0.0.1 --port", port)

print("Open the UI below and configure models in Start -> Video -> Model setup.")
print("Use the Model setup cards to install/check SAM3 Scene Sweep, choose SAM2 fallback, or link hosted providers.")
output.serve_kernel_port_as_iframe(port, path="/ui/", height=900)


In [ ]:
from google.colab import output

output.serve_kernel_port_as_window(8766, path="/ui/")


In [ ]:
subprocess.run([sys.executable, "examples/make_demo_video.py", "--out", "examples/demo_red_ball.mp4"], check=True)
print("Optional demo video path to register in the UI: examples/demo_red_ball.mp4")
print("Open the UI above, then use Video -> Add local video path if you want a sample clip.")


In [ ]:
if ui_proc.poll() is None and ui_proc.stdout is not None:
    print("UI server is still running. Recent logs will appear here only after the process writes more output.")
else:
    print("UI server exited with code", ui_proc.returncode)


In [ ]:
if 'ui_proc' in globals() and ui_proc.poll() is None:
    ui_proc.terminate()
    print("MotionJSON UI stopped.")


## Advanced fallback only

The cells below are not the normal setup flow. Use them only when the main UI's Model setup screen asks you to manually prepare local SAM paths or when you are debugging a Colab runtime.

Local SAM2 has two separate paths:

- `/content/sam2` is the cloned official SAM2 source/package directory. It lets Python import `sam2`, but it is not the checkpoint path to use in Model setup.
- `SAM2_LOCAL_CHECKPOINT` must be a real local checkpoint file path, usually `/content/sam2/checkpoints/sam2.1_hiera_large.pt` after you explicitly download checkpoints.
- `SAM2_LOCAL_CONFIG` must point to the matching local YAML config, usually `/content/sam2/sam2/configs/sam2.1/sam2.1_hiera_l.yaml`.

Local SAM3 concept/exemplar setup also has separate source and checkpoint paths:

- `/content/sam3` is the cloned official SAM3 source/package directory. It lets Python import `sam3`, but it is not a model checkpoint.
- `SAM3_LOCAL_MODEL` must be a real local checkpoint file path ending in `sam3.pt`.
- `facebook/sam3` is a gated Hugging Face repo id, not a local model path. Use of the local `facebook/sam3` model is allowed only after Meta has approved your access.


In [ ]:
print("Python:", sys.version.split()[0])
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("For local SAM2/SAM3, switch Colab to a GPU runtime before installing model packages.")
except Exception as exc:
    print("PyTorch is not importable yet:", type(exc).__name__)
    print("Hosted providers can still be linked. Local SAM setup needs torch plus the official SAM package.")


In [ ]:
import re
from getpass import getpass

SAM2_SOURCE_DIR = Path("/content/sam2")
SAM2_CHECKPOINT_FILENAME = "sam2.1_hiera_large.pt"
SAM2_CONFIG_FILENAME = "sam2.1_hiera_l.yaml"
SAM2_CONFIG_RELATIVE_PATH = Path("sam2/configs/sam2.1") / SAM2_CONFIG_FILENAME
SAM2_CHECKPOINT_DIR = SAM2_SOURCE_DIR / "checkpoints"
SAM2_DEFAULT_CONFIG_PATH = SAM2_SOURCE_DIR / SAM2_CONFIG_RELATIVE_PATH

SAM3_SOURCE_DIR = Path("/content/sam3")
SAM3_HF_REPO_ID = "facebook/sam3"
SAM3_CHECKPOINT_FILENAME = "sam3.pt"
SAM3_HF_CACHE_DIR = Path.home() / ".cache" / "huggingface" / "hub" / "models--facebook--sam3"

def colab_user_secret(name: str) -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return (value or "").strip()

def is_probably_hf_repo_id(value: str) -> bool:
    value = str(value or "").strip()
    return bool(re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", value)) and not value.startswith(("/", ".", "~"))

def friendly_size(path: Path) -> str:
    try:
        size = path.stat().st_size
    except OSError:
        return "unknown size"
    units = ["bytes", "KB", "MB", "GB", "TB"]
    value = float(size)
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:.1f} {unit}" if unit != "bytes" else f"{int(value)} bytes"
        value /= 1024
    return f"{size} bytes"

def find_named_file_candidates(paths: list[Path], filename: str) -> list[Path]:
    candidates: list[Path] = []
    seen: set[str] = set()
    for root in paths:
        root = Path(root).expanduser()
        if root.is_file() and root.name == filename:
            matches = [root]
        elif root.exists() and root.is_dir():
            matches = list(root.rglob(filename))
        else:
            matches = []
        for candidate in matches:
            if candidate.is_file():
                key = str(candidate.resolve())
                if key not in seen:
                    seen.add(key)
                    candidates.append(candidate)
    return sorted(candidates, key=lambda p: p.stat().st_mtime if p.exists() else 0, reverse=True)

def find_sam2_checkpoint_candidates(paths: list[Path]) -> list[Path]:
    return find_named_file_candidates(paths, SAM2_CHECKPOINT_FILENAME)

def find_sam2_config_candidates(paths: list[Path]) -> list[Path]:
    candidates = find_named_file_candidates(paths, SAM2_CONFIG_FILENAME)
    preferred = [path for path in candidates if str(SAM2_CONFIG_RELATIVE_PATH) in str(path)]
    return preferred or candidates

def find_sam3_checkpoint_candidates(paths: list[Path]) -> list[Path]:
    return find_named_file_candidates(paths, SAM3_CHECKPOINT_FILENAME)

def set_and_validate_sam2_local_checkpoint(path: str | Path) -> Path:
    raw = str(path or "").strip()
    if not raw:
        raise ValueError("SAM2_LOCAL_CHECKPOINT is empty. Use a local .pt checkpoint file path.")
    if is_probably_hf_repo_id(raw):
        raise ValueError(f"{raw!r} looks like a repo id, not a local SAM2 checkpoint path. Use the downloaded .pt file path.")
    checkpoint_path = Path(raw).expanduser()
    if str(checkpoint_path).rstrip("/") == str(SAM2_SOURCE_DIR):
        candidates = find_sam2_checkpoint_candidates([checkpoint_path])
        if candidates:
            raise ValueError(
                f"{checkpoint_path} is the cloned SAM2 source/package directory. "
                f"Use the checkpoint file instead: {candidates[0]}"
            )
        raise ValueError(
            f"{checkpoint_path} is the cloned SAM2 source/package directory, not the checkpoint. "
            "Run the SAM2 checkpoint resolver cell or paste a real .pt file path."
        )
    if checkpoint_path.is_dir():
        candidates = find_sam2_checkpoint_candidates([checkpoint_path])
        if candidates:
            print(f"{checkpoint_path} is a directory; using checkpoint file {candidates[0]}")
            checkpoint_path = candidates[0]
        else:
            raise ValueError(f"{checkpoint_path} is a directory and no {SAM2_CHECKPOINT_FILENAME} checkpoint was found inside it.")
    if not checkpoint_path.exists():
        raise ValueError(f"SAM2_LOCAL_CHECKPOINT path does not exist: {checkpoint_path}")
    if checkpoint_path.suffix != ".pt":
        print(f"Warning: expected a .pt checkpoint file; got {checkpoint_path.name}.")
    os.environ["SAM2_LOCAL_CHECKPOINT"] = str(checkpoint_path)
    return checkpoint_path

def set_and_validate_sam2_local_config(path: str | Path) -> Path:
    raw = str(path or "").strip()
    if not raw:
        raise ValueError("SAM2_LOCAL_CONFIG is empty. Use the matching local YAML config path.")
    config_path = Path(raw).expanduser()
    if str(config_path).rstrip("/") == str(SAM2_SOURCE_DIR):
        candidates = find_sam2_config_candidates([config_path])
        if candidates:
            raise ValueError(
                f"{config_path} is the cloned SAM2 source/package directory. "
                f"Use the config file instead: {candidates[0]}"
            )
        raise ValueError(f"{config_path} is the cloned SAM2 source/package directory, not the YAML config path.")
    if config_path.is_dir():
        candidates = find_sam2_config_candidates([config_path])
        if candidates:
            print(f"{config_path} is a directory; using config file {candidates[0]}")
            config_path = candidates[0]
        else:
            raise ValueError(f"{config_path} is a directory and no {SAM2_CONFIG_FILENAME} config was found inside it.")
    if not config_path.exists():
        raise ValueError(f"SAM2_LOCAL_CONFIG path does not exist: {config_path}")
    if config_path.suffix not in {".yaml", ".yml"}:
        print(f"Warning: expected a YAML config file; got {config_path.name}.")
    os.environ["SAM2_LOCAL_CONFIG"] = str(config_path)
    return config_path

def print_sam2_path_help(checkpoint_path: Path | None = None, config_path: Path | None = None) -> None:
    print("SAM2 local path guide:")
    print("- /content/sam2 is the official SAM2 source/package directory, not the checkpoint path.")
    print("- SAM2_LOCAL_CHECKPOINT must be a local .pt checkpoint file path.")
    print("- SAM2_LOCAL_CONFIG must be the matching local YAML config path.")
    if checkpoint_path and config_path:
        print("Copy these values into Model setup -> SAM2 fallback:")
        print("  provider: SAM2 local")
        print(f"  checkpoint path: {checkpoint_path}")
        print(f"  model config path: {config_path}")
        print(f"  device: {os.environ.get('SAM2_LOCAL_DEVICE', 'cuda')}")

def set_and_validate_sam3_local_model(path: str | Path) -> Path:
    raw = str(path or "").strip()
    if not raw:
        raise ValueError("SAM3_LOCAL_MODEL is empty. Use a local sam3.pt checkpoint file path.")
    if is_probably_hf_repo_id(raw):
        raise ValueError(
            f"{raw!r} is a Hugging Face repo id, not a local file path. "
            "Download or resolve facebook/sam3 sam3.pt first, then use the returned local path."
        )
    model_path = Path(raw).expanduser()
    if str(model_path).rstrip("/") == str(SAM3_SOURCE_DIR):
        candidates = find_sam3_checkpoint_candidates([model_path])
        if candidates:
            raise ValueError(
                f"{model_path} is the cloned SAM3 source/package directory. "
                f"Use the checkpoint file instead: {candidates[0]}"
            )
        raise ValueError(
            f"{model_path} is the cloned SAM3 source/package directory, not the checkpoint. "
            "Run the checkpoint resolver cell or paste a real sam3.pt file path."
        )
    if model_path.is_dir():
        candidates = find_sam3_checkpoint_candidates([model_path])
        if candidates:
            print(f"{model_path} is a directory; using checkpoint file {candidates[0]}")
            model_path = candidates[0]
        else:
            raise ValueError(f"{model_path} is a directory and no sam3.pt checkpoint was found inside it.")
    if not model_path.exists():
        raise ValueError(f"SAM3_LOCAL_MODEL path does not exist: {model_path}")
    if model_path.name != SAM3_CHECKPOINT_FILENAME:
        print(f"Warning: expected a file named {SAM3_CHECKPOINT_FILENAME}; got {model_path.name}.")
    os.environ["SAM3_LOCAL_MODEL"] = str(model_path)
    return model_path

def print_sam3_path_help(model_path: Path | None = None) -> None:
    print("SAM3 local path guide:")
    print("- /content/sam3 is the official SAM3 source/package directory, not the checkpoint path.")
    print("- facebook/sam3 is the Hugging Face repo id, not a local model path.")
    print("- Meta approval is required before using or downloading the local facebook/sam3 model checkpoint.")
    print("- No Hugging Face token is needed if you paste an approved local sam3.pt path from Google Drive or a manual upload.")
    print("- SAM3_LOCAL_MODEL must be a local checkpoint file path ending in sam3.pt.")
    if model_path:
        print("Copy these values into Model setup -> SAM3 Scene Sweep:")
        print(f"  provider: SAM3 local")
        print(f"  model path: {model_path}")
        print(f"  device: {os.environ.get('SAM3_LOCAL_DEVICE', 'cuda')}")

for env_name in ["ROBOFLOW_API_KEY", "REPLICATE_API_TOKEN", "FAL_KEY", "HF_TOKEN"]:
    value = colab_user_secret(env_name)
    if not value:
        value = getpass(f"{env_name} (leave blank to skip): ").strip()
    if value:
        os.environ[env_name] = value

if os.environ.get("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]

existing_sam2_checkpoint = colab_user_secret("SAM2_LOCAL_CHECKPOINT") or os.environ.get("SAM2_LOCAL_CHECKPOINT", "")
existing_sam2_config = colab_user_secret("SAM2_LOCAL_CONFIG") or os.environ.get("SAM2_LOCAL_CONFIG", "")
if existing_sam2_checkpoint or existing_sam2_config:
    try:
        resolved_sam2_checkpoint = set_and_validate_sam2_local_checkpoint(existing_sam2_checkpoint)
        resolved_sam2_config = set_and_validate_sam2_local_config(existing_sam2_config)
        print("SAM2 local paths are already configured and exist.")
        print_sam2_path_help(resolved_sam2_checkpoint, resolved_sam2_config)
    except ValueError as exc:
        os.environ.pop("SAM2_LOCAL_CHECKPOINT", None)
        os.environ.pop("SAM2_LOCAL_CONFIG", None)
        print("Existing SAM2 local paths are not usable:", exc)
        print_sam2_path_help()
else:
    print("No SAM2_LOCAL_CHECKPOINT/SAM2_LOCAL_CONFIG paths configured yet. Use the SAM2 checkpoint resolver cell after optional package setup.")

existing_sam3_path = colab_user_secret("SAM3_LOCAL_MODEL") or os.environ.get("SAM3_LOCAL_MODEL", "")
if existing_sam3_path:
    try:
        resolved_existing_path = set_and_validate_sam3_local_model(existing_sam3_path)
        print("SAM3_LOCAL_MODEL is already configured and exists.")
        print_sam3_path_help(resolved_existing_path)
    except ValueError as exc:
        os.environ.pop("SAM3_LOCAL_MODEL", None)
        print("Existing SAM3_LOCAL_MODEL is not usable:", exc)
        print_sam3_path_help()
else:
    print("No SAM3_LOCAL_MODEL path configured yet. Use the SAM3 checkpoint resolver cell after optional package setup.")

configured_secret_names = [name for name in ["ROBOFLOW_API_KEY", "REPLICATE_API_TOKEN", "FAL_KEY", "HF_TOKEN", "HUGGINGFACE_HUB_TOKEN"] if os.environ.get(name)]
configured_path_names = [name for name in ["SAM2_LOCAL_CHECKPOINT", "SAM2_LOCAL_CONFIG", "SAM3_LOCAL_MODEL"] if os.environ.get(name)]
print("Configured secret names (values hidden):", configured_secret_names or "none")
print("Configured local path names:", configured_path_names or "none")


## Advanced fallback: local SAM2 package setup

Use local SAM2 for `Trace one object` when you want point/box-prompted video segmentation inside this Colab runtime. Run this only after selecting a GPU runtime.

This cell installs the official SAM2 source package only. It clones `https://github.com/facebookresearch/sam2.git` into `/content/sam2` and runs `pip install -e /content/sam2`.

Important: `/content/sam2` is not the model checkpoint path. It is the source/package directory. Run the next cell to resolve or download a SAM2 checkpoint file and set `SAM2_LOCAL_CHECKPOINT` plus `SAM2_LOCAL_CONFIG`.


In [ ]:
RUN_LOCAL_SAM2_SETUP = False

if RUN_LOCAL_SAM2_SETUP:
    if not SAM2_SOURCE_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/sam2.git", str(SAM2_SOURCE_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(SAM2_SOURCE_DIR)], check=True)
    os.environ.setdefault("SAM2_LOCAL_DEVICE", "cuda")
    print(f"Installed SAM2 source/package from {SAM2_SOURCE_DIR}.")
    print("This installs SAM2 code only. It does not download checkpoint .pt files.")
    print_sam2_path_help()
else:
    print("Set RUN_LOCAL_SAM2_SETUP = True to clone and install the official SAM2 source package in this runtime.")
    print("This step does not download model checkpoints. Replicate SAM2 video can be linked from Model setup instead.")


## Advanced fallback: resolve or download the SAM2 checkpoint path

This cell finds or downloads the real checkpoint file for `SAM2_LOCAL_CHECKPOINT` and the matching YAML file for `SAM2_LOCAL_CONFIG`.

- `/content/sam2` is the source/package directory. It is useful for importing `sam2`, but it is not a checkpoint path to paste into MotionJSON.
- The value to use in Model setup is the local `.pt` checkpoint file, such as `/content/sam2/checkpoints/sam2.1_hiera_large.pt`.
- The config value should be the matching local YAML file, such as `/content/sam2/sam2/configs/sam2.1/sam2.1_hiera_l.yaml`.
- Downloads are opt-in. Leave `RUN_DOWNLOAD_SAM2_CHECKPOINTS = False` to avoid surprise large files.

If you already downloaded a SAM2 checkpoint, paste that file path into `MANUAL_SAM2_CHECKPOINT_PATH`. If the config is outside the usual SAM2 source tree, paste it into `MANUAL_SAM2_CONFIG_PATH`.


In [ ]:
RUN_DOWNLOAD_SAM2_CHECKPOINTS = False
MANUAL_SAM2_CHECKPOINT_PATH = ""  # Example: /content/sam2/checkpoints/sam2.1_hiera_large.pt
MANUAL_SAM2_CONFIG_PATH = ""      # Example: /content/sam2/sam2/configs/sam2.1/sam2.1_hiera_l.yaml

sam2_search_roots = [SAM2_CHECKPOINT_DIR, SAM2_SOURCE_DIR]
drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    sam2_search_roots.append(drive_root)

resolved_sam2_checkpoint: Path | None = None
resolved_sam2_config: Path | None = None
manual_checkpoint = MANUAL_SAM2_CHECKPOINT_PATH.strip()
manual_config = MANUAL_SAM2_CONFIG_PATH.strip()

if manual_checkpoint:
    resolved_sam2_checkpoint = set_and_validate_sam2_local_checkpoint(manual_checkpoint)
    print("Using manually supplied SAM2 checkpoint path.")
else:
    candidates = find_sam2_checkpoint_candidates(sam2_search_roots)
    if candidates:
        resolved_sam2_checkpoint = set_and_validate_sam2_local_checkpoint(candidates[0])
        print("Found cached SAM2 checkpoint:", resolved_sam2_checkpoint)
    elif RUN_DOWNLOAD_SAM2_CHECKPOINTS:
        if not SAM2_SOURCE_DIR.exists():
            raise RuntimeError("Clone/install the SAM2 source package first by running the optional SAM2 package setup cell.")
        download_script = SAM2_CHECKPOINT_DIR / "download_ckpts.sh"
        if not download_script.exists():
            raise RuntimeError(f"SAM2 checkpoint download script not found: {download_script}")
        print("Downloading official SAM2 checkpoints with download_ckpts.sh. This may download large files.")
        subprocess.run(["bash", "download_ckpts.sh"], cwd=str(SAM2_CHECKPOINT_DIR), check=True)
        candidates = find_sam2_checkpoint_candidates([SAM2_CHECKPOINT_DIR])
        if not candidates:
            raise RuntimeError(f"download_ckpts.sh finished but no {SAM2_CHECKPOINT_FILENAME} file was found in {SAM2_CHECKPOINT_DIR}.")
        resolved_sam2_checkpoint = set_and_validate_sam2_local_checkpoint(candidates[0])
    else:
        print(f"No cached {SAM2_CHECKPOINT_FILENAME} checkpoint was found in the usual Colab locations.")
        print("Set RUN_DOWNLOAD_SAM2_CHECKPOINTS = True only after accepting the large checkpoint download.")
        print("You can also paste an existing local .pt path into MANUAL_SAM2_CHECKPOINT_PATH.")
        print_sam2_path_help()

if manual_config:
    resolved_sam2_config = set_and_validate_sam2_local_config(manual_config)
    print("Using manually supplied SAM2 config path.")
else:
    config_candidates = find_sam2_config_candidates([SAM2_SOURCE_DIR, Path.cwd()])
    if config_candidates:
        resolved_sam2_config = set_and_validate_sam2_local_config(config_candidates[0])
        print("Found SAM2 config:", resolved_sam2_config)
    else:
        print(f"No {SAM2_CONFIG_FILENAME} config was found. Run the SAM2 package setup cell or paste a path into MANUAL_SAM2_CONFIG_PATH.")

if resolved_sam2_checkpoint and resolved_sam2_config:
    print("SAM2_LOCAL_CHECKPOINT set to:", resolved_sam2_checkpoint)
    print("Checkpoint size:", friendly_size(resolved_sam2_checkpoint))
    print("SAM2_LOCAL_CONFIG set to:", resolved_sam2_config)
    os.environ.setdefault("SAM2_LOCAL_DEVICE", "cuda")
    print_sam2_path_help(resolved_sam2_checkpoint, resolved_sam2_config)


## Advanced fallback: validate SAM2 local readiness

Run this before diagnosing local SAM2 in the UI. It checks runtime basics, package importability, CUDA, and whether `SAM2_LOCAL_CHECKPOINT` plus `SAM2_LOCAL_CONFIG` are real local files. It also calls MotionJSON backend diagnostics without making hosted provider calls.

Hosted SAM2 users can skip local readiness failures and configure Replicate SAM2 video or a custom SAM2-compatible endpoint in Model setup instead.


In [ ]:
from importlib.util import find_spec

if "readiness_row" not in globals():
    def readiness_row(label: str, ok: bool, detail: str) -> None:
        status = "OK" if ok else "CHECK"
        print(f"[{status}] {label}: {detail}")

torch_ok = False
cuda_ok = False
try:
    import torch
    torch_ok = True
    cuda_ok = bool(torch.cuda.is_available())
    detail = f"torch {torch.__version__}; CUDA available: {cuda_ok}"
    if cuda_ok:
        detail += f"; device: {torch.cuda.get_device_name(0)}"
    readiness_row("PyTorch/CUDA", cuda_ok, detail)
except Exception as exc:
    readiness_row("PyTorch/CUDA", False, f"torch import failed: {type(exc).__name__}: {exc}")

sam2_import_ok = find_spec("sam2") is not None
readiness_row("SAM2 package", sam2_import_ok, "Python can import sam2." if sam2_import_ok else "Install the official source package from /content/sam2 first.")

resolved_checkpoint_for_ui: Path | None = None
resolved_config_for_ui: Path | None = None
current_checkpoint_value = os.environ.get("SAM2_LOCAL_CHECKPOINT", "").strip()
current_config_value = os.environ.get("SAM2_LOCAL_CONFIG", "").strip()
try:
    resolved_checkpoint_for_ui = set_and_validate_sam2_local_checkpoint(current_checkpoint_value)
    readiness_row("SAM2_LOCAL_CHECKPOINT", True, f"{resolved_checkpoint_for_ui} ({friendly_size(resolved_checkpoint_for_ui)})")
except ValueError as exc:
    readiness_row("SAM2_LOCAL_CHECKPOINT", False, str(exc))
try:
    resolved_config_for_ui = set_and_validate_sam2_local_config(current_config_value)
    readiness_row("SAM2_LOCAL_CONFIG", True, str(resolved_config_for_ui))
except ValueError as exc:
    readiness_row("SAM2_LOCAL_CONFIG", False, str(exc))

if resolved_checkpoint_for_ui and resolved_config_for_ui:
    print_sam2_path_help(resolved_checkpoint_for_ui, resolved_config_for_ui)
else:
    print_sam2_path_help()

print("Running MotionJSON backend diagnostics without hosted network calls...")
subprocess.run([sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--text"], check=False)


## Advanced fallback: local SAM3 package setup

Use local SAM3 for concept prompts such as `red ball` or `person in white`. This cell installs the official SAM3 source package only. It clones `https://github.com/facebookresearch/sam3.git` into `/content/sam3` and runs `pip install -e /content/sam3`.

Important: `/content/sam3` is not the model checkpoint path. It is the source/package directory. Run the next cell to resolve or download the `facebook/sam3` checkpoint file and set `SAM3_LOCAL_MODEL`.

Use of the local `facebook/sam3` model is allowed only after Meta has approved your access to the gated model. The source package setup below does not grant model access and does not download model files.

SAM3 local setup expects official package/model access and may require a Python/CUDA combination that differs from the default Colab image. If local SAM3 is not ready, use Roboflow SAM3 or Fal SAM3 image from Model setup.


In [ ]:
RUN_LOCAL_SAM3_SETUP = False

if RUN_LOCAL_SAM3_SETUP:
    if sys.version_info < (3, 12):
        print(f"Warning: Python {sys.version.split()[0]} detected. Local SAM3 currently expects Python 3.12+.")
    if not SAM3_SOURCE_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/facebookresearch/sam3.git", str(SAM3_SOURCE_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(SAM3_SOURCE_DIR)], check=True)
    os.environ.setdefault("SAM3_LOCAL_DEVICE", "cuda")
    print(f"Installed SAM3 source/package from {SAM3_SOURCE_DIR}.")
    print("This installs SAM3 code only. It does not download facebook/sam3 sam3.pt.")
    print_sam3_path_help()
else:
    print("Set RUN_LOCAL_SAM3_SETUP = True to clone and install the official SAM3 source package in this runtime.")
    print("This step does not download model checkpoints. Hosted Roboflow SAM3 or Fal SAM3 image can be linked instead.")


## Advanced fallback: resolve or download the SAM3 checkpoint path

This cell finds or downloads the real checkpoint file for `SAM3_LOCAL_MODEL`.

- `facebook/sam3` is a Hugging Face repo id. It is useful for `hf_hub_download`, but it is not a local path to paste into MotionJSON.
- Meta approval is required before using or downloading the gated `facebook/sam3` model checkpoint. Only set `RUN_DOWNLOAD_SAM3_CHECKPOINT = True` after your Hugging Face account has been approved by Meta for this model.
- If you do not want to configure Hugging Face API tokens in Colab, use the Google Drive/manual path instead: put an already-approved `sam3.pt` at `GOOGLE_DRIVE_SAM3_CHECKPOINT_PATH` or paste a local file path into `MANUAL_SAM3_CHECKPOINT_PATH`. This avoids Hugging Face token setup, but it does not bypass Meta approval.
- If you do not want local gated-model setup at all, skip this cell and use Roboflow SAM3 or Fal SAM3 image in Model setup.
- The value to use in Model setup is the local `sam3.pt` file path, usually either the Google Drive/manual path you provide or the Hugging Face cache path returned by `hf_hub_download`.
- Downloads are opt-in. Leave `RUN_DOWNLOAD_SAM3_CHECKPOINT = False` to avoid surprise large files.

If you already downloaded `sam3.pt`, paste that file path into `MANUAL_SAM3_CHECKPOINT_PATH` and run the cell.


In [ ]:
RUN_DOWNLOAD_SAM3_CHECKPOINT = False
RUN_USE_GOOGLE_DRIVE_SAM3_CHECKPOINT = False
GOOGLE_DRIVE_SAM3_CHECKPOINT_PATH = "/content/drive/MyDrive/motionjson-models/sam3.pt"
MANUAL_SAM3_CHECKPOINT_PATH = ""  # Example: /content/drive/MyDrive/motionjson-models/sam3.pt

search_roots = [SAM3_HF_CACHE_DIR, SAM3_SOURCE_DIR]
drive_checkpoint_path = Path(GOOGLE_DRIVE_SAM3_CHECKPOINT_PATH).expanduser()
if drive_checkpoint_path.parent.exists():
    search_roots.append(drive_checkpoint_path.parent)

resolved_model_path: Path | None = None
manual_path = MANUAL_SAM3_CHECKPOINT_PATH.strip()

if manual_path:
    resolved_model_path = set_and_validate_sam3_local_model(manual_path)
    print("Using manually supplied SAM3 checkpoint path.")
    print("No Hugging Face token is needed for a manual local path, but the sam3.pt file must still come from a Meta-approved access path.")
elif RUN_USE_GOOGLE_DRIVE_SAM3_CHECKPOINT:
    try:
        from google.colab import drive
    except Exception:
        drive = None
    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.exists():
        if drive is None:
            raise RuntimeError("Google Drive mounting is only available in Colab. Paste a local sam3.pt path into MANUAL_SAM3_CHECKPOINT_PATH instead.")
        drive.mount("/content/drive")
    print("Using Google Drive SAM3 checkpoint path. No Hugging Face token is required for this path.")
    print("Only use a sam3.pt file that Meta has approved you to access and use.")
    resolved_model_path = set_and_validate_sam3_local_model(drive_checkpoint_path)
else:
    candidates = find_sam3_checkpoint_candidates(search_roots)
    if candidates:
        resolved_model_path = set_and_validate_sam3_local_model(candidates[0])
        print("Found cached SAM3 checkpoint:", resolved_model_path)
    elif RUN_DOWNLOAD_SAM3_CHECKPOINT:
        token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
        if not token:
            raise RuntimeError("HF_TOKEN or HUGGINGFACE_HUB_TOKEN is required after Meta approves your Hugging Face access for facebook/sam3. To avoid Hugging Face tokens, use MANUAL_SAM3_CHECKPOINT_PATH or RUN_USE_GOOGLE_DRIVE_SAM3_CHECKPOINT with an approved sam3.pt file.")
        try:
            from huggingface_hub import hf_hub_download
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub"], check=True)
            from huggingface_hub import hf_hub_download
        print("Downloading facebook/sam3 sam3.pt with hf_hub_download. This may be a large gated file.")
        print("Only continue if Meta has approved your access to facebook/sam3. The notebook cannot bypass gated model approval.")
        print("The Hugging Face token is passed directly and is not printed.")
        downloaded_path = hf_hub_download(repo_id=SAM3_HF_REPO_ID, filename=SAM3_CHECKPOINT_FILENAME, token=token)
        resolved_model_path = set_and_validate_sam3_local_model(downloaded_path)
    else:
        print("No cached sam3.pt checkpoint was found in the usual Colab locations.")
        print("Set RUN_DOWNLOAD_SAM3_CHECKPOINT = True only after Meta approves Hugging Face access and you accept the large download.")
        print("To avoid Hugging Face token setup, set RUN_USE_GOOGLE_DRIVE_SAM3_CHECKPOINT = True after placing an approved sam3.pt at GOOGLE_DRIVE_SAM3_CHECKPOINT_PATH.")
        print("You can also paste an existing approved local sam3.pt path into MANUAL_SAM3_CHECKPOINT_PATH.")
        print("Or skip local SAM3 and use Roboflow SAM3 or Fal SAM3 image in Model setup.")
        print_sam3_path_help()

if resolved_model_path:
    print("SAM3_LOCAL_MODEL set to:", resolved_model_path)
    print("Checkpoint size:", friendly_size(resolved_model_path))
    print_sam3_path_help(resolved_model_path)


## Advanced fallback: validate SAM3 local readiness

Run this before diagnosing local SAM3 in the UI. It checks runtime basics, package importability, CUDA, and whether `SAM3_LOCAL_MODEL` is a real local checkpoint path. It also calls MotionJSON backend diagnostics without making hosted provider calls.

Hosted SAM3 users can skip local readiness failures and configure Roboflow SAM3 or Fal SAM3 image in Model setup instead. Local SAM3 users still need Meta approval for the gated `facebook/sam3` model checkpoint.


In [ ]:
from importlib.util import find_spec

def readiness_row(label: str, ok: bool, detail: str) -> None:
    status = "OK" if ok else "CHECK"
    print(f"[{status}] {label}: {detail}")

py_ok = sys.version_info >= (3, 12)
readiness_row("Python", py_ok, f"{sys.version.split()[0]} detected; local SAM3 expects Python 3.12+.")

torch_ok = False
cuda_ok = False
try:
    import torch
    torch_ok = True
    cuda_ok = bool(torch.cuda.is_available())
    detail = f"torch {torch.__version__}; CUDA available: {cuda_ok}"
    if cuda_ok:
        detail += f"; device: {torch.cuda.get_device_name(0)}"
    readiness_row("PyTorch/CUDA", cuda_ok, detail)
except Exception as exc:
    readiness_row("PyTorch/CUDA", False, f"torch import failed: {type(exc).__name__}: {exc}")

sam3_import_ok = find_spec("sam3") is not None
readiness_row("SAM3 package", sam3_import_ok, "Python can import sam3." if sam3_import_ok else "Install the official source package from /content/sam3 first.")

resolved_for_ui: Path | None = None
current_model_value = os.environ.get("SAM3_LOCAL_MODEL", "").strip()
try:
    resolved_for_ui = set_and_validate_sam3_local_model(current_model_value)
    readiness_row("SAM3_LOCAL_MODEL", True, f"{resolved_for_ui} ({friendly_size(resolved_for_ui)})")
except ValueError as exc:
    readiness_row("SAM3_LOCAL_MODEL", False, str(exc))

if resolved_for_ui:
    print_sam3_path_help(resolved_for_ui)
else:
    print_sam3_path_help()

print("Running MotionJSON backend diagnostics without hosted network calls...")
subprocess.run([sys.executable, "-m", "motionjson.cli", "backend", "diagnostics", "--text"], check=False)
